# 2AFC Psychophysics Statistics Pipeline (mixed-design ANOVA)

**Self-contained, additive analysis.** This notebook does **not** recompute
psychometric fits and does **not** modify any existing analysis code. It reads
a *frozen* per-subject x finger psychometric summary and produces the full
statistics pipeline: validation/QC, an exploratory one-way ANOVA, the **main**
mixed-design ANOVA, planned contrasts, a subject-respecting bootstrap,
publication figures, and a methods/results report.

## Domain facts (used verbatim where reported)
- 2AFC stiffness-discrimination task. Standard **S = 85**; comparison values
  25, 40, 55, 70, 100, 115, 130, 145; predictor **delta = C - 85**.
- The binary response is coded **y = 1 iff the participant judged the
  COMPARISON stiffer than the standard** (not correct/incorrect, not
  side/order, not physical truth). This coding was done upstream.

## Psychometric model (frozen upstream)
Each subject x finger function is a **lapse-aware yes/no-style** function with
**four fitted parameters** (`mu`, `scale`, `lapse_low`, `lapse_high`) - *not* a
textbook 2AFC percent-correct model with a fixed guess rate. It models
P(y=1):

$$P(y{=}1) = \text{lapse\_low} + (1 - \text{lapse\_low} - \text{lapse\_high})\,F(\delta;\ \mu, \text{scale}),\quad \delta = C - 85,$$

with $F$ a monotonic sigmoid. Fits used **psignifit** when installed
(preferred) and otherwise a **custom lapse-aware fallback fitter**; the fitter
per row is recorded in `fit_method` / `psignifit_status`.

## Derived measures (in comparison-stiffness units = delta-space)
- **PSE** = comparison value where P(y=1) = 0.5.
- **Bias** = PSE - 85 = the delta at P(y=1) = 0.5 (perceptual shift; 0 = none).
- **JND** = (x75 - x25)/2, where x25/x75 are the comparison values where the
  curve reaches 0.25/0.75 (sensitivity; smaller = finer). Because of the lapse
  terms, x25/x75 are estimable only if 0.25 and 0.75 lie within
  [lapse_low, 1 - lapse_high]; otherwise the upstream fitter returns NaN JND
  with a fit warning, so a non-estimable JND is **flagged**, not silently
  invalid.

## Design
- **System** (L vs N) = **between-subject** factor (each subject uses one
  system).
- **Finger** (I, M, P, R) = **within-subject / repeated** factor (4 per
  subject).
- **Subject** = repeated-measures unit.

## Why a mixed-design ANOVA and NOT an ordinary two-way ANOVA
An ordinary independent two-way ANOVA assumes all observations are
independent. The four finger measurements come from the **same** subject and
are therefore correlated (repeated measures); treating them as independent
would violate the independence assumption and mis-estimate the error term. The
mixed-design model (System between, Finger within, Subject as the repeated
unit) is the correct classical model and is the **MAIN** analysis.


## 0. Setup and configuration

We use `pingouin.mixed_anova` for the main analysis (it reports df, F, p,
partial eta-squared, generalized eta-squared, Mauchly sphericity and the
Greenhouse-Geisser correction in one call). A fixed random seed governs **all**
stochastic steps (bootstrap and figure jitter).

In [ ]:
# Ensure this notebook's selected kernel has the required packages.
# This fixes `ModuleNotFoundError: No module named 'pandas'` when VS Code/Jupyter
# opens the notebook with a Python environment that has not been prepared yet.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'scipy': 'scipy',
    'statsmodels': 'statsmodels',
    'pingouin': 'pingouin',
    'openpyxl': 'openpyxl',
}

missing = [package for module, package in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing notebook packages into:', sys.executable)
    print('Missing:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    importlib.invalidate_caches()
else:
    print('Notebook dependencies are available in:', sys.executable)


In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import importlib, sys
import anova_statistics as A
# Re-sync with the latest code on disk so a long-lived kernel never runs a
# stale module (prevents 'module has no attribute ...' after edits without a
# kernel restart). Reload the colocated oneway_anova helper first.
if 'oneway_anova' in sys.modules:
    importlib.reload(sys.modules['oneway_anova'])
A = importlib.reload(A)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("pingouin", A.pg.__version__)
print("Random seed (all stochastic steps):", A.RANDOM_SEED)

# Output uses a FLAT taxonomy: each analysis type is one folder; the cohort is
# encoded in the FILENAME prefix (``<cohort>__...``). This notebook's interactive
# cells analyse the full (unfiltered) cohort, so they write the ``raw`` cohort
# (``raw__`` prefixes). Section 8 adds the filtered subgroups as sibling cohorts
# in the SAME flat folders; section 9 adds the one-way per-system analysis.
RESULTS = "results"  # notebook runs from the anova_statistics directory
COHORT = "raw"
P = A._ensure_pipeline_dirs(RESULTS, COHORT)   # creates mixed_design/, bootstrap/, report/raw, summary_table/raw
MIXED, BOOT = P["mixed"], P["boot"]
REPORTS, TABLES = P["reports"], P["tables"]
# Per-DV sub-folders (bias/jnd) under each analysis root; cohort stays in filename.
def mixed_dir(dv, kind):
    d = os.path.join(MIXED, "bias" if dv in ("Bias", "PSE") else "jnd", kind)
    os.makedirs(d, exist_ok=True); return d
def boot_dir(dv, kind):
    d = os.path.join(BOOT, "bias" if dv in ("Bias", "PSE") else "jnd", kind)
    os.makedirs(d, exist_ok=True); return d
print("Results root:", os.path.abspath(RESULTS), "| cohort:", COHORT)





## 1. Load and validate

We read the **live** psychophysics summary directly from the results tree:
`analysis/psychophysics/results/L_N_E/_working/pse_jnd_by_subject_finger.csv`.

> **No recomputation.** PSE, JND and Bias are read **as-is** from this
> already-computed summary; psychometric fits are never re-run here. (A legacy
> frozen copy under `data/` is kept only as an offline fallback if the live
> results folder is unavailable, in which case `load_data` warns.)

Validation prints clear warnings and **never silently drops data**. It checks:
each Subject in exactly one System; no duplicate Subject x Finger rows; required
columns; System has 2 levels; Finger has 4 levels; Bias == PSE - 85 (within
tol); missing Bias/JND; balance; flagged-fit counts; and extreme/non-positive
JND.

In [ ]:
df = A.load_data()          # frozen copy by default
print("Source path:", df.attrs["source_path"])
print("Documented provenance:", A.SOURCE_DATA_PATH)
print("Shape:", df.shape)

validation = A.validate_data(df)
print(validation.render())


### 1a. Per-Subject x Finger QC flags

Explicit QC flags derived from the **existing** fit columns (no refitting):
estimability of PSE/x25/x75/JND, whether lapse parameters were pinned at a
bound (0 or 0.20), total lapse rate, non-positive / extreme JND, the fitter
used, and `fit_quality` / `fit_warning`. `qc_pass` marks rows with estimable
PSE & JND, a positive non-extreme JND, and `fit_quality == 'ok'`.

In [ ]:
df_flags = A.compute_qc_flags(df)
qc_summary = A.qc_flag_summary(df_flags)
qc_summary


### 1b. Included vs excluded datasets (transparency CSVs)

The upstream pipeline flags some fits with `excluded_from_group_analysis`.
We save **both** the included and excluded sets (the excluded set carries an
`exclusion_reason` column) plus the full QC-flagged table. The **MAIN** ANOVA
still runs on **all valid rows**; a **sensitivity** analysis later excludes the
flagged fits.

In [ ]:
included, excluded = A.split_included_excluded(df)
print(f"Included rows: {len(included)}   Excluded (flagged) rows: {len(excluded)}")

qc_dir = REPORTS  # results/reports/raw
df_flags.to_csv(os.path.join(qc_dir, "qc_flags_all_rows.csv"), index=False)
qc_summary.to_csv(os.path.join(qc_dir, "qc_flag_summary.csv"), index=False)
included.to_csv(os.path.join(qc_dir, "included_rows.csv"), index=False)
if len(excluded):
    excluded.to_csv(os.path.join(qc_dir, "excluded_rows.csv"), index=False)
print("Saved QC + included/excluded CSVs to", os.path.abspath(qc_dir))
if len(excluded):
    display(excluded[["Subject","System","Finger","PSE","Bias","JND",
                      "fit_quality","exclusion_reason"]])

### 1c. Descriptive summaries

Mean / SD / SE / N per System x Finger, for Bias and JND. (JND descriptives are
strongly affected by a handful of degenerate fits - see the QC flags above and
the sensitivity analysis below.)

In [ ]:
desc_bias = A.descriptive_table(df, "Bias")
desc_jnd  = A.descriptive_table(df, "JND")
display(desc_bias.round(3))
display(desc_jnd.round(3))


## 2. Exploratory one-way ANOVA on the 8 combined System x Finger groups

We form `Group8 = System + "_" + Finger` (8 groups) and run a one-way ANOVA,
separately for Bias and JND, with Holm-corrected pairwise post-hoc tests.

> **This is EXPLORATORY only.** It treats the 8 cells as independent groups and
> does **NOT** model the repeated-measures structure. It is reported for
> completeness and is **not** the basis for inference.

In [ ]:
exploratory = {}
for dv in ["Bias", "JND"]:
    aov, posthoc = A.exploratory_one_way(df, dv)
    exploratory[dv] = {"aov": aov, "posthoc": posthoc}
    aov.to_csv(os.path.join(mixed_dir(dv, "csv"),
                            f"{COHORT}__exploratory_oneway_anova.csv"), index=False)
    posthoc.to_csv(os.path.join(mixed_dir(dv, "csv"),
                               f"{COHORT}__exploratory_oneway_posthoc_holm.csv"),
                   index=False)
    print(f"--- Exploratory one-way ANOVA on combined System x Finger groups: {dv} ---")
    display(aov.round(4))


In [ ]:
# Exploratory 8-group figures (points + mean +/- 95% CI; labelled exploratory).
A.plot_eight_groups(df, "Bias", "Bias (PSE - 85)",
                    "Exploratory: Bias by combined System_Finger group",
                    os.path.join(mixed_dir("Bias", "figures"),
                                 f"{COHORT}__exploratory_8group.png"), hline=0.0)
A.plot_eight_groups(df, "JND", "JND",
                    "Exploratory: JND by combined System_Finger group",
                    os.path.join(mixed_dir("JND", "figures"),
                                 f"{COHORT}__exploratory_8group.png"), hline=None)
from IPython.display import Image, display
display(Image(os.path.join(mixed_dir("Bias", "figures"), f"{COHORT}__exploratory_8group.png")))
display(Image(os.path.join(mixed_dir("JND", "figures"), f"{COHORT}__exploratory_8group.png")))


## 3. MAIN analysis: mixed-design ANOVA

System (between) x Finger (within), Subject = repeated unit, separately for
Bias and JND. We report the main effect of **System**, the main effect of
**Finger**, the **System x Finger interaction**, df, F, p, partial eta-squared
(`np2`), generalized eta-squared (`ges` when available), Mauchly's test of
sphericity, and the Greenhouse-Geisser-corrected p (`p_GG_corr`) for the Finger
factor.

**Method:** `pingouin.mixed_anova` (chosen because it returns the mixed-design
F-tests, effect sizes, and sphericity/GG correction in one validated call). The
main analysis uses **all valid rows**; a **sensitivity** analysis (flagged fits
excluded) follows.

In [ ]:
mixed = {}
mixed_sensitivity = {}
for dv in ["Bias", "JND"]:
    mixed[dv] = A.mixed_anova(df, dv, drop_flagged=False)            # MAIN
    mixed_sensitivity[dv] = A.mixed_anova(df, dv, drop_flagged=True) # SENSITIVITY
    mixed[dv]["aov"].to_csv(os.path.join(mixed_dir(dv, "csv"),
                            f"{COHORT}__mixed_anova_main.csv"), index=False)
    mixed_sensitivity[dv]["aov"].to_csv(os.path.join(mixed_dir(dv, "csv"),
                            f"{COHORT}__mixed_anova_sensitivity.csv"), index=False)
    print(f"===== MAIN mixed-design ANOVA: {dv} "
          f"(N subjects = {mixed[dv]['n_subjects']}) =====")
    display(mixed[dv]["aov"].round(4))
    print("Sphericity (Finger):", mixed[dv]["sphericity_note"])
    print()
    print(f"----- SENSITIVITY (flagged fits removed): {dv} "
          f"(N subjects = {mixed_sensitivity[dv]['n_subjects']}) -----")
    display(mixed_sensitivity[dv]["aov"].round(4))


In [ ]:
# Main figures: DV by Finger, hue System, subject points + mean +/- 95% CI.
A.plot_dv_by_finger(df, "Bias", 0.0, "Bias (PSE - 85)",
                    "Bias by Finger and System (mean +/- 95% CI)",
                    os.path.join(mixed_dir("Bias", "figures"),
                                 f"{COHORT}__by_finger_system.png"))
A.plot_dv_by_finger(df, "PSE", A.STANDARD_VALUE, "PSE",
                    "PSE by Finger and System (reference = standard 85)",
                    os.path.join(mixed_dir("PSE", "figures"),
                                 f"{COHORT}__pse_raw_by_finger_system.png"))
A.plot_dv_by_finger(df, "JND", None, "JND",
                    "JND by Finger and System (mean +/- 95% CI)",
                    os.path.join(mixed_dir("JND", "figures"),
                                 f"{COHORT}__by_finger_system.png"))
from IPython.display import Image, display
for p in [os.path.join(mixed_dir("Bias", "figures"), f"{COHORT}__by_finger_system.png"),
          os.path.join(mixed_dir("PSE", "figures"), f"{COHORT}__pse_raw_by_finger_system.png"),
          os.path.join(mixed_dir("JND", "figures"), f"{COHORT}__by_finger_system.png")]:
    display(Image(p))


## 4. Planned contrasts / simple effects: System L vs N within each finger

Because subjects differ across systems, the L-vs-N comparison within a finger
is a **between-groups (independent)** test (Welch's t-test). We Holm-correct
across the 4 finger comparisons. The table reports per-system mean +/- SD, the
L - N difference, t, df, raw p, Holm p, the 95% CI of the difference, and
Cohen's d.

In [ ]:
contrasts = {}
for dv in ["Bias", "JND"]:
    ct = A.planned_contrasts(df, dv, drop_flagged=False)
    contrasts[dv] = ct
    ct.to_csv(os.path.join(mixed_dir(dv, "csv"),
                           f"{COHORT}__planned_contrasts_L_vs_N.csv"), index=False)
    print(f"--- Planned contrasts (System L - N within finger; Holm): {dv} ---")
    display(ct.round(4))
    A.plot_contrasts(ct, dv, f"{dv}",
                     f"Planned contrast: {dv} difference (L - N) per finger",
                     os.path.join(mixed_dir(dv, "figures"),
                                  f"{COHORT}__planned_contrast.png"))
from IPython.display import Image, display
display(Image(os.path.join(mixed_dir("Bias", "figures"), f"{COHORT}__planned_contrast.png")))
display(Image(os.path.join(mixed_dir("JND", "figures"), f"{COHORT}__planned_contrast.png")))


## 5. Bootstrap robustness (subject-level, structure-preserving)

We resample **subjects with replacement within each System**, keeping each
subject's 4 finger rows together (this respects the repeated-measures
structure). We compute bootstrap 95% CIs for (a) the mean per System x Finger
and (b) the L - N difference per finger, for both Bias and JND, with a fixed
seed.

In [ ]:
bootstrap = {}
for dv in ["Bias", "JND"]:
    cell_ci, diff_ci = A.bootstrap_cis(df, dv, n_boot=A.N_BOOTSTRAP,
                                       seed=A.RANDOM_SEED, drop_flagged=False)
    bootstrap[dv] = {"cell": cell_ci, "diff": diff_ci}
    cell_ci.to_csv(os.path.join(boot_dir(dv, "csv"),
                                f"{COHORT}__bootstrap_cell_means.csv"), index=False)
    diff_ci.to_csv(os.path.join(boot_dir(dv, "csv"),
                                f"{COHORT}__bootstrap_LminusN_diff.csv"), index=False)
    print(f"--- Bootstrap cell means (System x Finger): {dv} ---")
    display(cell_ci.round(3))
    print(f"--- Bootstrap L - N difference per finger: {dv} ---")
    display(diff_ci.round(3))


In [ ]:
# Bootstrap figures: L - N difference per finger (each finger in its colour).
from IPython.display import Image, display
for dv in ["Bias", "JND"]:
    out = os.path.join(boot_dir(dv, "figures"), f"{COHORT}__bootstrap_LminusN.png")
    A.plot_bootstrap_diff(bootstrap[dv]["diff"], dv, out)
    display(Image(out))


## 6. Methods + results report

We assemble and save the full methods/results report. It includes the
response-coding statement, the lapse-aware model and definitions, the
exploratory-one-way caveat, the mixed-design rationale (an ordinary independent
two-way ANOVA was **not** used because the 4 finger measurements per subject
are repeated measures), and the actual numeric results. Non-significant System
effects are reported as *"No significant evidence for a difference between
systems was observed"* - never as the systems being identical.

In [ ]:
# Run the one-way per-system analysis FIRST so the report can include it.
oneway = A.run_oneway_flat(base_root=RESULTS)
if oneway.get("note"):
    print("One-way note:", oneway["note"])
else:
    print("One-way per-system analysis written for systems:", oneway["systems"])

results = {
    "data_path": df.attrs["source_path"],
    "qc_summary": qc_summary,
    "exploratory": exploratory,
    "mixed": mixed,
    "mixed_sensitivity": mixed_sensitivity,
    "contrasts": contrasts,
    "bootstrap": bootstrap,
    "oneway": oneway,
}
report_txt = A.build_report(validation, results)
print(report_txt)

with open(os.path.join(REPORTS, "anova_statistics_report.txt"), "w",
          encoding="utf-8") as fh:
    fh.write(report_txt)
with open(os.path.join(REPORTS, "anova_statistics_report.md"), "w",
          encoding="utf-8") as fh:
    fh.write("```\n" + report_txt + "\n```\n")
print("\nSaved report to", os.path.abspath(REPORTS))


## 7. Summary of outputs

Outputs are organised by analysis type; the cohort/system/subject is encoded in
the **filename prefix**. `mixed_design/` and `bootstrap/` split by DV (`bias`/`jnd`):

- `results/mixed_design/{bias,jnd}/{csv,figures}` - mixed-ANOVA (main +
  sensitivity), the folded-in exploratory one-way 8-group tables/figure, and the
  planned-contrast tables/figures. Files are `<cohort>__...` (e.g.
  `mixed_design/bias/csv/raw__mixed_anova_main.csv`).
- `results/bootstrap/{bias,jnd}/{csv,figures}` - bootstrap CIs/figures, `<cohort>__...`.
- `results/oneway/subjects/{csv,figures}` - per-subject one-way outputs,
  `<system>_<subject>__...` (section 9).
- `results/oneway_filters/{csv,figures}` - per-system one-way ANOVA + Friedman
  tables/figures, `<system>__...` (section 9).
- `results/summary_table/<cohort>/` - the summary **table images** (incl. the
  one-way per-system summary for the raw cohort).
- `results/report/<cohort>/` - QC CSVs, included/excluded sets, the
  methods/results report (`.txt` + `.md`), `pipeline_notes.txt`, and (for
  subgroups) `subgroup_membership.csv`.
- `results/subgroup_master_summary.{csv,png}` - the cross-cohort comparison.

**Reminder for the manuscript.** The main analysis is the mixed-design ANOVA;
the one-way 8-group analysis is exploratory and ignores repeated measures.
Report non-significant System effects cautiously (no evidence of a difference,
not equivalence). The JND measure is sensitive to a small number of degenerate
fits (extreme JND); the sensitivity analysis (flagged fits removed), the
subject-level bootstrap, **and the subgroup sensitivity ladder (section 8)**
guard against those outliers driving conclusions - across every cleaning rule
tested, the System effect remained non-significant.



## 8. Subgroup sensitivity analyses (filtered cohorts)

The raw analysis above keeps **all** subjects. Here we re-run the **entire
pipeline** on filtered cohorts to check whether any conclusion depends on a
handful of poor performers or degenerate fits. Each filter **excludes bad
subjects** at three escalating strictness levels; filtering is at the
**subject** level (a subject's four finger rows are kept or dropped together,
so the repeated-measures design stays balanced).

**Filters** (a *finger* is "bad" if it crosses the threshold; the *level* says
how many bad fingers remove the whole subject):

| Filter | A finger is "bad" when | Source |
|---|---|---|
| `success_lt70` | success rate (% correct) `< 0.70` | **precomputed** `success_summary_by_subject_finger.csv` |
| `pse_shift_gt100` | &#124;PSE &minus; 85&#124; `> 100` | live summary |
| `jnd_gt250` | JND `> 250` | live summary |
| `jnd_gt100` | JND `> 100` | live summary |

**Levels:** `all_fingers` (exclude only if *all 4* are bad &mdash; loosest) &rarr;
`at_least_2_fingers` &rarr; `at_least_1_finger` (exclude if *any* finger is bad
&mdash; strictest).

> **No recomputation.** Success rate is **read** from the psychophysics
> pipeline's precomputed per-finger summary (`success_rate = n_correct /
> n_trials`); it is not re-derived from trials here.

**Output layout.** `mixed_design/` and `bootstrap/` split by DV (`bias`/`jnd`);
each cohort &mdash; `raw` plus each `<filter>__<level>` subgroup &mdash; is a
`<cohort>__` filename prefix within those (no per-cohort sub-folders):

```
results/
  mixed_design/{bias,jnd}/{csv,figures}   <cohort>__mixed_anova_main.csv, ...exploratory..., ...planned_contrast...
  bootstrap/{bias,jnd}/{csv,figures}      <cohort>__bootstrap_cell_means.csv, ...
  report/<cohort>/                        (report, QC, pipeline_notes, subgroup_membership)
  summary_table/<cohort>/                 (table images)
  subgroup_master_summary.{csv,png}
```

Each subgroup's `report/<cohort>/subgroup_membership.csv` lists exactly which
subjects were dropped and why. A cohort too small for the model (`< 4` subjects
or `< 2` per system) is **skipped with a recorded note** rather than crashed.
The **master comparison image** collects the mixed-ANOVA p-values across the raw
cohort and every cleaning rule, so you can see at a glance whether the System /
Finger / interaction conclusions are stable.



In [ ]:
# Per-finger success rate (% correct) is READ from the psychophysics pipeline's
# precomputed summary (not recomputed). Then run the COMPLETE pipeline on every
# filtered subgroup as its own cohort, written analysis-first under
# results/<analysis>/<filter>__<level>/. The already-computed raw results are
# passed in so the master comparison includes the unfiltered cohort WITHOUT
# recomputing it.
success_rates = A.load_success_rates()
print("Per-finger success rates:", success_rates.shape[0],
      "subject x finger rows; fingers < 70%:",
      int((success_rates["success_rate"] < A.SUCCESS_RATE_THRESHOLD).sum()))

master, subgroup_results = A.run_all_subgroups(
    df, base_root=RESULTS, success_rates=success_rates,
    raw_results=results, n_bootstrap=A.N_BOOTSTRAP, seed=A.RANDOM_SEED)

display(master[["cohort", "n_excluded", "n_subjects", "status",
                "Bias_System_p", "JND_System_p"]])

from IPython.display import Image, display
print("\nMaster cross-cohort comparison (mixed-ANOVA p-values):")
display(Image(os.path.join(RESULTS, "subgroup_master_summary.png")))
print("Each cohort's outputs live under results/<analysis>/<cohort>/ ;",
      "membership records under results/reports/<cohort>/.")

In [ ]:
# Render every summary as a publication-style table IMAGE (values baked in).
# Significant rows (p < .05) and assumption violations are shaded for the eye.
# TABLES == results/summary_table/raw (defined in the setup cell).
table_paths = A.render_summary_tables(df, results, TABLES)

# One-way per-system summary table image (raw cohort).
oneway_table = A.render_oneway_summary_table(oneway, TABLES)
if oneway_table:
    table_paths["oneway_per_system_summary"] = oneway_table

from IPython.display import Image, display
for name in ["mixed_anova_summary", "mixed_anova_sensitivity",
             "descriptives", "planned_contrasts", "assumption_checks",
             "oneway_per_system_summary"]:
    if name in table_paths:
        print(f"--- {name} ---")
        display(Image(table_paths[name]))
print("Saved summary-table images to", os.path.abspath(TABLES))


## 9. One-way per-system analysis (numpy/scipy)

A lightweight, repeated-measures-respecting complement to the mixed-design
ANOVA: **for each system separately**, do the four fingers differ? Two tests
per DV are run on the SAME frozen per-subject x finger summary:

- **One-way ANOVA** (`scipy.stats.f_oneway`) with eta^2 / omega^2 effect sizes
  (fingers treated as independent groups);
- **Friedman test** (`scipy.stats.friedmanchisquare`) with Kendall's W
  (repeated-measures, robust to the normality violations seen here).

Holm-corrected pairwise post-hocs (Welch *t* parametric, Wilcoxon signed-rank
non-parametric) and Shapiro/Levene assumption checks accompany each test.

**Flat output layout** (reusing `oneway_anova.py (colocated)`):

- `results/oneway_filters/{csv,figures}` - one set **per system** (`L`, `N`),
  `<system>__...` (test summary, descriptives, assumptions, post-hocs, figures).
- `results/oneway/subjects/{csv,figures}` - one set **per subject**,
  `<system>_<subject>__...` (per-subject CSV + Bias/JND figure).

The cell below was already executed in section 6 (`A.run_oneway_flat`) so the
report could include it; here we just display the per-system summaries inline.


In [ ]:
# Display the one-way per-system test summaries + figures (already computed in
# section 6 as ``oneway = A.run_oneway_flat(...)``). Re-run here if needed.
try:
    oneway
except NameError:
    oneway = A.run_oneway_flat(base_root=RESULTS)

from IPython.display import Image, display
if oneway.get("note"):
    print(oneway["note"])
else:
    for system in oneway["systems"]:
        print(f"===== One-way per-system summary: System {system} =====")
        display(oneway["test_summaries"][system].round(4))
        for dv in ["bias", "jnd"]:
            p = os.path.join(RESULTS, "oneway_filters", "figures",
                             f"{system}__{dv}_by_finger.png")
            if os.path.exists(p):
                display(Image(p))
    print("One-way outputs: results/oneway_filters/{csv,figures} (per system),",
          "results/oneway/subjects/{csv,figures} (per subject).")
